# EmpowerLens — Kaggle GPU runner

Runs real transformer fine-tunes on Kaggle's free GPU, then makes `results_combined/` downloadable.

**Before running:**
1. Settings → **Accelerator: GPU**, and **Internet: On** (needed for `pip` + model downloads from Hugging Face).
2. The splits in `data/splits_combined/` must already be **committed and pushed** on the branch below — the notebook uses them as-is and never regenerates them.

**This run trains DeBERTa-v3 only** (MentalRoBERTa already succeeded and is commented out
in `MODELS` below) on the `multilabel` task, 3 seeds each, so it lands
side-by-side in the same `results_combined/paper_comparison.csv`:

| tag | HF model id | what it is |
|---|---|---|
| `roberta-base` | `roberta-base` | current baseline (already have numbers for this) |
| `deberta-v3-base` | `microsoft/deberta-v3-base` | general-domain, disentangled attention — best "architecture swap" candidate |
| `mental-roberta-base` | `mental/mental-roberta-base` | RoBERTa continued-pretrained on Reddit mental-health corpora |

Set `MODELS` below to skip `roberta-base` if you already have those checkpoints/results.

This notebook only orchestrates shell commands; all logic lives in `src/`.

In [ ]:
# 1. Clone the repo and install the transformer stack.
REPO_URL = "https://github.com/lumia-Qcode/EmpowerLens.git"
BRANCH   = "lumia-space"

!rm -rf empowerlens && git clone --branch $BRANCH $REPO_URL empowerlens
%cd empowerlens
!pip install -q -r requirements-transformer.txt
# DeBERTa-v3's tokenizer is SentencePiece-based; make sure it's present.
!pip install -q sentencepiece protobuf

In [ ]:
# 1a. mental/mental-roberta-base is a GATED model on the Hub — you must (1) accept its
#     terms at https://huggingface.co/mental/mental-roberta-base while logged in, then
#     (2) add a Kaggle Secret named HF_TOKEN (Add-ons -> Secrets, top menu) holding a
#     Hugging Face access token with read scope. Without this, training that model 401s.
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=hf_token)
    print("Logged in to Hugging Face Hub.")
except Exception as e:
    print(f"[warn] no working HF_TOKEN secret ({e}) — mental/mental-roberta-base will 401 "
          f"until you accept its terms and add the Kaggle secret.")

In [ ]:
# 1b. Show exactly what train.csv is made of before spending GPU hours on it —
#     n_annotated vs n_codipas rows and the per-class breakdown from make_splits_combined.py's manifest.
import json
manifest_path = "data/splits_combined/split_manifest.json"
try:
    m = json.load(open(manifest_path))
    print(f"train = {m['n_train_annotated']} Annotated + {m['n_train_codipas']} CODIPAS "
          f"= {m['n_train_combined']} rows | val={m['n_val']} test={m['n_test']} (frozen Annotated benchmark)")
    print("\nper-class (combined train):")
    for cls, n in m["train_per_class_combined"].items():
        print(f"  {cls:22s} {n}")
except FileNotFoundError:
    print(f"[warn] {manifest_path} not found — did you push data/splits_combined/ to {BRANCH}?")

In [ ]:
# 2. Choose ONE task, train it over three seeds for EACH model, evaluate each on val+test,
#    then aggregate everything at the end. --device auto resolves to the Kaggle GPU (cuda).
TASK   = "multilabel"  # one of: binary | multiclass | multilabel
SEEDS  = (42, 1337, 2024)
MODELS = [
    "microsoft/deberta-v3-base",
    # "mental/mental-roberta-base",   # already trained successfully — commented out for this run
]

# DeBERTa-v3's disentangled attention uses more memory per layer than RoBERTa's plain
# attention at the same batch size; it OOM'd at the default 16 on this GPU. Halve it here
# instead of touching the CLI default, so other models keep their normal batch size.
BATCH_SIZE = {
    "microsoft/deberta-v3-base": 8,
    "mental/mental-roberta-base": 16,
}

SPLITS_DIR = "data/splits_combined"

OUT_DIR = "results_combined"
!mkdir -p $OUT_DIR   # created up front so later cells never hit a missing-directory error

for model in MODELS:
    tag = model.split("/")[-1]
    bs = BATCH_SIZE.get(model, 16)
    print(f"\n=== {model} (batch_size={bs}) ===")
    for seed in SEEDS:
        ckpt = f"checkpoints/{TASK}_{tag}_{seed}"
        !python -m src.train_transformer --task $TASK --model $model --seed $seed --device auto --splits $SPLITS_DIR --batch-size $bs
        !python -m src.evaluate --checkpoint $ckpt --reference --splits $SPLITS_DIR --out $OUT_DIR

!python -m src.aggregate --results $OUT_DIR


In [ ]:
# 4. Compare against the existing results/ (roberta-base, original Annotated_data splits)
#    and results_codipas/ (CODIPAS-only splits) already committed in the repo, alongside
#    this run's results_combined/ (DeBERTa-v3 this run + MentalRoBERTa from the prior run,
#    both on combined splits — both land in the same results_combined/paper_comparison.csv).
import pandas as pd
from pathlib import Path

SOURCES = {"results": "results", "results_codipas": "results_codipas", "results_combined": OUT_DIR}

frames = []
for label, folder in SOURCES.items():
    p = Path(folder) / "paper_comparison.csv"
    if p.exists():
        d = pd.read_csv(p)
        d["results_dir"] = label
        frames.append(d)
    else:
        print(f"[skip] {p} not found")

all_results = pd.concat(frames, ignore_index=True)
view = all_results[(all_results["task"] == TASK) & (all_results["split"] == "test")]
comparison = (
    view.groupby(["results_dir", "model"])[["weighted_f1", "macro_f1"]]
    .agg(["mean", "std"]).round(3)
)
print(comparison)

all_results.to_csv(f"{OUT_DIR}/all_sources_comparison.csv", index=False)
print(f"\nWrote combined comparison table to {OUT_DIR}/all_sources_comparison.csv")

In [ ]:
# 5. Copy results_combined/ to the Kaggle output so it can be downloaded from the session.
!mkdir -p /kaggle/working/$OUT_DIR
!cp -r $OUT_DIR/* /kaggle/working/$OUT_DIR/
!ls -la /kaggle/working/$OUT_DIR